# SILVER-CUSTOMERS

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

In [ ]:
df_customers = spark.table("ecommerce_lakehouse.bronze.customers")
df_orders = spark.table("ecommerce_lakehouse.silver.orders")

In [0]:
df_orders.limit(1).display()

In [0]:
df_customers.count()

In [0]:
df_customers.select(countDistinct("customer_id").alias("unique_count")).display()

In [0]:
df_customers.select(countDistinct("customer_unique_id").alias("unique_count")).display()

### Step 1: Dedup

In [0]:
# one row per PERSON: keep their most recent order's attributes
w = Window.partitionBy("customer_id").orderBy(lit(1))
seed = (df_customers
    .withColumn("rn", row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

In [0]:
seed.printSchema()

### Step 2: Casting

In [0]:
seed = seed.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))
seed.printSchema()

### Step 3: NULL check

In [0]:
seed = seed.filter(col("customer_id").isNotNull())
seed.count()

In [0]:
seed.limit(2).display()

### Step 4: write to Silver table

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_lakehouse.silver")

In [ ]:
seed.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("ecommerce_lakehouse.silver.customers")
spark.table("ecommerce_lakehouse.silver.customers").count()

In [ ]:
spark.table("ecommerce_lakehouse.silver.customers").limit(2).display()